# Section 05c Onboarding Notebook: Periodic Electronic Structure — Crystal Lattices, Bloch Orbitals, $\mathbf{k}$-Point `KRHF` & Band Structures

### Overview & Pedagogical Design
In extended crystalline solids, polymers, and 2D materials, translational symmetry replaces a finite molecule with an infinite Bravais lattice of unit cells indexed by lattice vectors $\mathbf{R}_L = n_1 \mathbf{a}_1 + n_2 \mathbf{a}_2 + n_3 \mathbf{a}_3$. By **Bloch's Theorem**, eigenstates of a periodic Hamiltonian can be labeled by a crystal momentum wavevector $\mathbf{k}$ in the First Brillouin Zone (FBZ). In atom-centered Gaussian basis sets (used by [`pyscf.pbc`](https://pyscf.org/user/pbc/scf.html), `CP2K`, and `CRYSTAL`), local atomic orbitals $\chi_\mu(\mathbf{r} - \mathbf{R}_L)$ are summed into translational-symmetry-adapted **Bloch Atomic Orbitals**:

$$\phi_\mu^{(\mathbf{k})}(\mathbf{r}) = \frac{1}{\sqrt{N_{\mathbf{k}}}} \sum_{\mathbf{R}_L} e^{i \mathbf{k} \cdot \mathbf{R}_L}\,\chi_\mu(\mathbf{r} - \mathbf{R}_L)$$

This block-diagonalizes the infinite crystal Roothaan–Hall equations into coupled $\mathbf{k}$-point eigenvalue equations:

$$\mathbf{F}(\mathbf{k})\,\mathbf{C}(\mathbf{k}) = \mathbf{S}(\mathbf{k})\,\mathbf{C}(\mathbf{k})\,\boldsymbol{\varepsilon}(\mathbf{k})$$

Once the periodic self-consistent field (`KRHF` / `KDFT`) converges on a uniform **Monkhorst–Pack $\mathbf{k}$-mesh**, we reconstruct the real-space density matrix $\mathbf{D}(\mathbf{R}_L)$ and evaluate $\mathbf{F}(\mathbf{k}_{\text{band}})$ along high-symmetry paths ($\Gamma \to X \to M \to \Gamma$) to produce the **electronic band structure** $\varepsilon_n(\mathbf{k})$ and **Density of States (DOS)** ([PySCF `get_bands` Guide](https://pyscf.org/user/pbc/scf.html)).

In this notebook, **no periodic electronic structure functions are pre-solved for you**. You will implement **14 granular `# TODO` functions**:
- **Part 1 — Crystal Lattices, Reciprocal Vectors & Monkhorst–Pack $\mathbf{k}$-Grids (`TODO 1` – `TODO 3`)**
- **Part 2 — Bloch Lattice Sums, $\mathbf{k}$-Point Roothaan–Hall Diagonalization & Brillouin-Zone Fourier Transforms (`TODO 4` – `TODO 6`)**
- **Part 3 — Periodic Crystalline Hartree–Fock (`KRHF`), Exchange Divergence & Band Structure Calculation (`TODO 7` – `TODO 13`)**
- **Part 4 — Production `pyscf.pbc` Script Generator (`TODO 14`) & Step-by-Step Checkpoint Suite (`run_section05c_checkpoints`)**

In [ ]:
import numpy as np
from scipy.linalg import eigh
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True, linewidth=110)
HARTREE_TO_EV = 27.211386245988
print("Section 05c Periodic Electronic Structure environment initialized. Implement TODOs 1-14 and run `run_section05c_checkpoints()`!")

## Part 1: Crystal Lattices, Reciprocal Space & Monkhorst–Pack $\mathbf{k}$-Point Grids (TODOs 1–3)

### Mathematical Specifications
1. **Reciprocal Lattice Vectors & Unit Cell Volume (`compute_reciprocal_lattice_and_volume` — TODO 1)**:
   Given the $3 \times 3$ real-space lattice matrix $\mathbf{A}$ whose rows are the primitive lattice vectors $\mathbf{a}_1, \mathbf{a}_2, \mathbf{a}_3$ (in Bohr):
   $$\Omega = |\det(\mathbf{A})|, \qquad \mathbf{B} = 2\pi \left(\mathbf{A}^{-1}\right)^\mathsf{T} \quad \Longrightarrow \quad \mathbf{A}\mathbf{B}^\mathsf{T} = 2\pi \mathbf{I}_3$$
   The rows of $\mathbf{B}$ are the reciprocal lattice vectors $\mathbf{b}_1, \mathbf{b}_2, \mathbf{b}_3$ (in $\text{Bohr}^{-1}$).

2. **Uniform Monkhorst–Pack Brillouin-Zone Grid (`generate_monkhorst_pack_kmesh` — TODO 2)**:
   For a grid of $(N_1, N_2, N_3)$ $\mathbf{k}$-points ($N_{\mathbf{k}} = N_1 N_2 N_3$), the centered Monkhorst–Pack fractional coordinates along axis $d \in \{1, 2, 3\}$ are:
   $$u_{d, r} = \frac{2r - N_d - 1}{2 N_d}, \qquad r = 1, 2, \dots, N_d \quad \left(u_{d, r} \in \left[-\tfrac{1}{2}, +\tfrac{1}{2}\right)\right)$$
   Each fractional vector $\mathbf{u} = (u_1, u_2, u_3)$ maps to Cartesian reciprocal space via $\mathbf{k} = \mathbf{u}\,\mathbf{B}$ with uniform integration weight $w_{\mathbf{k}} = \frac{1}{N_{\mathbf{k}}}$.

3. **High-Symmetry $\mathbf{k}$-Path Interpolation (`interpolate_high_symmetry_kpath` — TODO 3)**:
   Given an ordered list of high-symmetry $\mathbf{k}$-points $[\mathbf{K}_0, \mathbf{K}_1, \dots, \mathbf{K}_M]$ (e.g., $\Gamma \to X \to M \to \Gamma$), linearly interpolate `n_per_segment` points along each segment and compute the cumulative 1D path distance $s_m = s_{m-1} + \|\mathbf{k}_m - \mathbf{k}_{m-1}\|_2$ for band-structure plotting (together with the tick locations `tick_positions` of the high-symmetry vertices).

In [ ]:
def compute_reciprocal_lattice_and_volume(
    a_vecs: np.ndarray,
) -> tuple[np.ndarray, float]:
    '''
    TODO 1: Given real-space lattice matrix `a_vecs` of shape (3, 3) (rows are a_1, a_2, a_3 in Bohr),
    return (b_vecs, cell_volume_bohr3), where rows of `b_vecs` are reciprocal vectors b_1, b_2, b_3.
    '''
    # TODO 1: Compute volume = abs(det(a_vecs)) and b_vecs = 2 * pi * inv(a_vecs).T.
    raise NotImplementedError("Complete TODO 1: compute_reciprocal_lattice_and_volume")


def generate_monkhorst_pack_kmesh(
    b_vecs: np.ndarray, n_kpts: tuple[int, int, int] = (8, 1, 1)
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    '''
    TODO 2: Generate a centered Monkhorst-Pack k-point grid of shape (N_k, 3) where N_k = n1 * n2 * n3.
    Return (k_frac, k_cart, k_weights) of shapes (N_k, 3), (N_k, 3), and (N_k,).
    '''
    # TODO 2: Construct 1D grids u_d = (2*np.arange(1, N_d+1) - N_d - 1) / (2*N_d) for d in 0..2,
    #         form the Cartesian product of shape (N_k, 3), compute k_cart = k_frac @ b_vecs,
    #         and set k_weights = np.full(N_k, 1.0 / N_k).
    raise NotImplementedError("Complete TODO 2: generate_monkhorst_pack_kmesh")


def interpolate_high_symmetry_kpath(
    high_sym_kpts_cart: list[np.ndarray], n_per_segment: int = 25
) -> tuple[np.ndarray, np.ndarray, list[float]]:
    '''
    TODO 3: Interpolate a continuous path through `high_sym_kpts_cart` with `n_per_segment` points
    per segment (avoiding duplicate endpoints at internal vertices).
    Return (kpath_cart, kpath_dist, vertex_tick_distances).
    '''
    # TODO 3: Linearly interpolate between each pair (K_m, K_{m+1}), accumulate Euclidean step lengths
    #         into kpath_dist, and record the distance at each high-symmetry vertex.
    raise NotImplementedError("Complete TODO 3: interpolate_high_symmetry_kpath")

## Part 2: Bloch Lattice Sums, $\mathbf{k}$-Point Roothaan–Hall Solver & Real-Space Density Reconstruction (TODOs 4–6)

### Mathematical Specifications
1. **Real-Space to Bloch Basis Lattice Sum (`bloch_lattice_sum_matrix` — TODO 4)**:
   Given real-space operator matrices $\mathbf{M}(\mathbf{R}_L)$ of shape `(N_cells, n_ao, n_ao)` for lattice translation vectors $\mathbf{R}_L$ of shape `(N_cells, 3)`, the Bloch matrix at wavevector $\mathbf{k}$ is:
   $$\mathbf{M}(\mathbf{k}) = \sum_{L=0}^{N_{\text{cells}}-1} e^{i \mathbf{k} \cdot \mathbf{R}_L}\,\mathbf{M}(\mathbf{R}_L)$$
   Because $\mathbf{M}(-\mathbf{R}_L) = \mathbf{M}(\mathbf{R}_L)^\mathsf{T}$ for real AOs, $\mathbf{M}(\mathbf{k})$ is guaranteed to be complex Hermitian ($\mathbf{M}(\mathbf{k}) = \mathbf{M}(\mathbf{k})^\dagger$).

2. **Complex Hermitian $\mathbf{k}$-Point Roothaan–Hall Solver (`solve_kpoint_roothaan_hall` — TODO 5)**:
   At each $\mathbf{k}$-point, diagonalize the complex Hermitian overlap matrix $\mathbf{S}(\mathbf{k}) = \mathbf{U}(\mathbf{k})\,\mathbf{s}(\mathbf{k})\,\mathbf{U}(\mathbf{k})^\dagger$, retain eigenvalues $s_\alpha(\mathbf{k}) > \text{svd\_tol}$ to eliminate crystal linear dependencies ([PySCF PBC Linear Dependency Guide](https://pyscf.org/user/pbc/scf.html)), form $\mathbf{X}(\mathbf{k}) = \mathbf{U}_{\text{keep}}(\mathbf{k})\,\mathbf{s}_{\text{keep}}(\mathbf{k})^{-1/2}\,\mathbf{U}_{\text{keep}}(\mathbf{k})^\dagger$, solve $\mathbf{F}'(\mathbf{k})\mathbf{C}'(\mathbf{k}) = \mathbf{C}'(\mathbf{k})\boldsymbol{\varepsilon}(\mathbf{k})$, back-transform $\mathbf{C}(\mathbf{k}) = \mathbf{X}(\mathbf{k})\mathbf{C}'(\mathbf{k})$, and construct the $\mathbf{k}$-point spatial density matrix:
   $$D_{\mu\nu}(\mathbf{k}) = \sum_{i=0}^{n_{\text{occ}}-1} C_{\mu i}(\mathbf{k})\,C_{\nu i}^*(\mathbf{k}) \quad \Longleftrightarrow \quad \mathbf{D}(\mathbf{k}) = \mathbf{C}_{\text{occ}}(\mathbf{k})\,\mathbf{C}_{\text{occ}}(\mathbf{k})^\dagger$$

3. **Inverse Fourier Transform to Real-Space Density Matrix (`reconstruct_real_space_density_from_kpts` — TODO 6)**:
   $$\mathbf{D}(\mathbf{R}_L) = \sum_{\mathbf{k}} w_{\mathbf{k}}\,e^{-i \mathbf{k} \cdot \mathbf{R}_L}\,\mathbf{D}(\mathbf{k})$$

In [ ]:
def bloch_lattice_sum_matrix(
    real_space_blocks: np.ndarray,
    lattice_translations: np.ndarray,
    k_vec: np.ndarray,
) -> np.ndarray:
    '''
    TODO 4: Compute M(k) = sum_L exp(i k . R_L) M(R_L) from `real_space_blocks` of shape (n_cells, n_ao, n_ao)
    and `lattice_translations` of shape (n_cells, 3), and Hermitianize 0.5 * (M_k + M_k.conj().T).
    '''
    # TODO 4: Compute phases = np.exp(1j * (lattice_translations @ k_vec)), contract with real_space_blocks, and Hermitianize.
    raise NotImplementedError("Complete TODO 4: bloch_lattice_sum_matrix")


def solve_kpoint_roothaan_hall(
    F_k: np.ndarray, S_k: np.ndarray, n_occ: int, svd_tol: float = 1e-7
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    '''
    TODO 5: Solve the complex Hermitian generalized eigenvalue problem F(k) C(k) = S(k) C(k) eps(k)
    at a single k-point and return (eps_k, C_k, D_k), where D_k = C_occ(k) @ C_occ(k)^dagger.
    '''
    # TODO 5: Diagonalize S_k with eigh, build X_k filtering s > svd_tol, diagonalize X_k^dagger F_k X_k,
    #         back-transform C_k = X_k @ C_prime_k, and form D_k = C_k[:, :n_occ] @ C_k[:, :n_occ].conj().T.
    raise NotImplementedError("Complete TODO 5: solve_kpoint_roothaan_hall")


def reconstruct_real_space_density_from_kpts(
    D_kpts: np.ndarray,
    k_vecs: np.ndarray,
    lattice_translations: np.ndarray,
    k_weights: np.ndarray,
) -> np.ndarray:
    '''
    TODO 6: Reconstruct real-space density matrix blocks D(R_L) = Re[ sum_k w_k exp(-i k . R_L) D(k) ]
    of shape (n_cells, n_ao, n_ao) from `D_kpts` of shape (n_kpts, n_ao, n_ao).
    '''
    # TODO 6: Compute phase matrix exp(-1j * (lattice_translations @ k_vecs.T)) and contract with k_weights * D_kpts.
    raise NotImplementedError("Complete TODO 6: reconstruct_real_space_density_from_kpts")

## Part 3: Periodic Crystalline Hartree–Fock (`KRHF`), Exchange Divergence & Band Structure (TODOs 7–13)

### Mathematical Specifications
We model a 1D crystalline polymer / semiconductor (such as *trans*-polyacetylene $\text{-(CH)}_2\text{-}_\infty$ or a dimerized covalent chain with unit cell length $a$ and bond-length alternation $\delta$) having **2 atomic orbitals per unit cell** ($\mu \in \{A, B\}$, $N_{\text{elec}}^{\text{cell}} = 2$, $n_{\text{occ}} = 1$ filled valence band and $1$ empty conduction band):
- **Intracell ($R_L = 0$) bond**: distance $d_1 = \frac{a}{2} - \delta$, overlap $s_1 = s_0 e^{-\alpha(d_1 - a/2)}$, hopping $t_1 = -\beta_0 e^{-\alpha(d_1 - a/2)}$, Coulomb repulsion $V_{AB}(0) = v_{\text{nn}} e^{-\alpha(d_1 - a/2)}$, on-site repulsion $U_{AA}(0) = U_{BB}(0) = U_0$.
- **Intercell ($R_L = -a$) bond** (connecting site $B$ in cell $0$ to site $A$ in cell $+1$, i.e., $H_{AB}(R_L = +a)$ and $H_{BA}(R_L = -a)$): distance $d_2 = \frac{a}{2} + \delta$, overlap $s_2 = s_0 e^{-\alpha(d_2 - a/2)}$, hopping $t_2 = -\beta_0 e^{-\alpha(d_2 - a/2)}$, intercell Coulomb $V_{AB}(+a) = v_{\text{nn}} e^{-\alpha(d_2 - a/2)}$.
- **Why Hartree–Fock Opens the Peierls Band Gap**: In periodic `KRHF`, the nonlocal Fock exchange matrix element across the short vs. long bond is $K_{\mu\nu}(R_L) = V_{\mu\nu}(R_L)\,D_{\mu\nu}(R_L)$, which self-consistently amplifies the bond-order alternation $D_{AB}(0) > D_{AB}(+a)$! Furthermore, the $\mathbf{G}=\mathbf{0}$ Madelung constant (`exxdiv`) shifts the occupied band energies by $-\frac{1}{2}v_{\text{Madelung}} P_{\mu\mu}(0)$ ([PySCF PBC Exchange Divergence](https://pyscf.org/user/pbc/scf.html)).

In [ ]:
def build_peierls_chain_real_space_tensors(
    a_lat: float = 4.60,
    delta_dimer: float = 0.15,
    beta_0: float = 0.095,
    s_0: float = 0.18,
    alpha_decay: float = 1.10,
    e_onsite: float = -0.45,
    u_onsite: float = 0.32,
    v_nn: float = 0.16,
) -> dict[str, np.ndarray | float]:
    '''
    TODO 7: Construct the real-space overlap S(R_L), core Hamiltonian H_core(R_L), and two-center
    Coulomb kernel V_ee(R_L) for R_L in [-a_lat, 0.0, +a_lat] (3 cells: L = -1, 0, +1).
    Return a dict with keys 'lattice_translations' (3, 3), 'S_R' (3, 2, 2), 'H_core_R' (3, 2, 2),
    'V_ee_R' (3, 2, 2), 'a_vecs' (3, 3), 'e_nuc_cell' (float).
    '''
    # TODO 7: Compute short-bond (d1 = a_lat/2 - delta_dimer) and long-bond (d2 = a_lat/2 + delta_dimer)
    #         couplings (t1, s1, v1) and (t2, s2, v2):
    #         - L = 0 (R_L = 0): S_0 = [[1, s1], [s1, 1]], H_0 = [[e_onsite, t1], [t1, e_onsite]], V_0 = [[u_onsite, v1], [v1, u_onsite]]
    #         - L = +1 (R_L = +a_lat x): S_{+1}[0, 1] = s2, H_{+1}[0, 1] = t2, V_{+1}[0, 1] = v2
    #         - L = -1 (R_L = -a_lat x): transpose of L = +1 blocks so M(-R_L) == M(R_L).T!
    raise NotImplementedError("Complete TODO 7: build_peierls_chain_real_space_tensors")


def build_kpoint_fock_matrix(
    k_vec: np.ndarray,
    H_core_R: np.ndarray,
    V_ee_R: np.ndarray,
    D_R: np.ndarray,
    lattice_translations: np.ndarray,
    madelung_const: float = 0.0,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    '''
    TODO 8: Construct the real-space Coulomb J(R_L) and Exchange K(R_L) blocks from D_R and V_ee_R,
    transform H_core(k), J(k), and K(k) to k-space via `bloch_lattice_sum_matrix`, and return
    (F_k, H_core_k, JK_k) where F_k = H_core_k + 2*J_k - K_k.
    Specifically:
    - Real-space Hartree diagonal J(R_L=0)_{mu,mu} = sum_{L, nu} V_ee_R[L, mu, nu] * D_R[1, nu, nu] (where index 1 is R_L=0)
    - Real-space nonlocal Exchange K(R_L)_{mu,nu} = V_ee_R[L, mu, nu] * D_R[L, mu, nu]
      plus madelung_const * D_R[1, mu, mu] on the R_L=0 diagonal.
    '''
    # TODO 8: Assemble J_R and K_R of shape (n_cells, n_ao, n_ao), call bloch_lattice_sum_matrix for H_core_k and F_k,
    #         and return (F_k, H_core_k, 2*J_k - K_k).
    raise NotImplementedError("Complete TODO 8: build_kpoint_fock_matrix")


def compute_krhf_energy_per_unit_cell(
    D_kpts: np.ndarray,
    H_core_kpts: np.ndarray,
    F_kpts: np.ndarray,
    k_weights: np.ndarray,
    e_nuc_cell: float = 0.0,
) -> tuple[float, float]:
    '''
    TODO 9: Compute the crystalline RHF electronic energy and total energy per unit cell:
    E_elec = sum_k w_k * Re(Tr[D(k) @ (H_core(k) + F(k))]), E_tot = E_elec + e_nuc_cell.
    Return (e_elec_cell, e_tot_cell).
    '''
    # TODO 9: Contract D_kpts with (H_core_kpts + F_kpts) weighted by k_weights.
    raise NotImplementedError("Complete TODO 9: compute_krhf_energy_per_unit_cell")


def run_periodic_krhf_scf(
    model: dict,
    n_kpts_1d: int = 12,
    n_occ: int = 1,
    madelung_const: float = 0.0,
    max_iter: int = 60,
    tol: float = 1e-10,
) -> dict:
    '''
    TODO 10: Run the self-consistent periodic KRHF cycle on a (n_kpts_1d, 1, 1) Monkhorst-Pack mesh
    using TODOs 1, 2, 4, 5, 6, 8, and 9.
    Return a dict with keys:
        'converged', 'e_tot_cell', 'e_elec_cell', 'D_R', 'D_kpts', 'eps_kpts',
        'k_cart', 'k_weights', 'n_iterations', 'energy_history'.
    '''
    # TODO 10: Generate k-mesh, initialize D_R from core guess (F_k = H_core_k), and iterate
    #          F_k -> D_k -> D_R -> E_tot_cell until |dE| < tol.
    raise NotImplementedError("Complete TODO 10: run_periodic_krhf_scf")


def compute_band_structure_along_kpath(
    kpath_cart: np.ndarray,
    model: dict,
    D_R_converged: np.ndarray,
    madelung_const: float = 0.0,
) -> np.ndarray:
    '''
    TODO 11: Non-self-consistently evaluate and diagonalize F(k) along `kpath_cart` (shape N_path x 3)
    using the converged real-space density matrix `D_R_converged` (analogous to PySCF's `kmf.get_bands(kpts)`).
    Return `bands_eh` of shape (N_path, n_ao).
    '''
    # TODO 11: For each k_vec in kpath_cart, build F_k (TODO 8) and S_k (TODO 4), solve Roothaan-Hall (TODO 5),
    #          and collect the eigenvalues eps(k).
    raise NotImplementedError("Complete TODO 11: compute_band_structure_along_kpath")


def analyze_band_gap_and_effective_mass(
    kx_path: np.ndarray, bands_eh: np.ndarray, n_occ: int = 1
) -> dict[str, float | bool]:
    '''
    TODO 12: From 1D wavevector grid `kx_path` (in Bohr^-1) and `bands_eh` (shape N_path x n_bands, in Hartree):
    1. Find the Valence Band Maximum (VBM = max of band n_occ - 1) and Conduction Band Minimum (CBM = min of band n_occ).
    2. Compute fundamental gap E_g in Hartree and eV, and check whether is_direct_gap (idx_vbm == idx_cbm).
    3. Estimate the conduction-band effective mass m*_cb = 1.0 / (d^2 eps_cb / dk^2) at idx_cbm using a
       3-point central finite difference stencil (or one-sided stencil if idx_cbm is on the boundary).
    Return a dict with keys 'vbm_eh', 'cbm_eh', 'gap_eh', 'gap_ev', 'is_direct_gap', 'k_vbm', 'k_cbm', 'm_eff_cb'.
    '''
    # TODO 12: Extract VBM, CBM, gap_eh, gap_ev, is_direct_gap, and second-derivative effective mass m_eff_cb.
    raise NotImplementedError("Complete TODO 12: analyze_band_gap_and_effective_mass")


def compute_electronic_density_of_states(
    eps_kpts_eh: np.ndarray,
    k_weights: np.ndarray,
    energy_grid_ev: np.ndarray,
    sigma_ev: float = 0.25,
) -> np.ndarray:
    '''
    TODO 13: Compute the spin-summed Gaussian-broadened Electronic Density of States DOS(E) (states / eV / cell)
    on `energy_grid_ev` from `eps_kpts_eh` (shape N_k x n_bands, in Hartree) and `k_weights`.
    '''
    # TODO 13: Convert eps_kpts_eh to eV, evaluate Gaussian kernels (1 / (sqrt(2*pi)*sigma)) * exp(-0.5*((E - eps)/sigma)^2),
    #          and sum over bands and k-points weighted by 2.0 * k_weights.
    raise NotImplementedError("Complete TODO 13: compute_electronic_density_of_states")


def generate_pyscf_pbc_krhf_script(
    a_lat_ang: float = 2.46, kmesh: tuple[int, int, int] = (8, 1, 1)
) -> str:
    '''
    TODO 14: Return a complete Python script string that uses `pyscf.pbc.gto.Cell` and
    `pyscf.pbc.scf.KRHF` (with `kmf.get_bands(kpts_band)`) to run a periodic KRHF band-structure calculation.
    Must include `pbc.gto.Cell`, `make_kpts`, `KRHF`, and `get_bands`.
    '''
    # TODO 14: Format and return the PySCF periodic KRHF + get_bands script string.
    raise NotImplementedError("Complete TODO 14: generate_pyscf_pbc_krhf_script")

## Part 4: Step-by-Step Checkpoint & Verification Suite

Run `run_section05c_checkpoints()` below as you implement each `# TODO` function to verify your reciprocal lattice builder, Monkhorst–Pack $\mathbf{k}$-mesh, Bloch lattice sums, periodic `KRHF` solver, Peierls band gap Opening, effective mass analyzer, and DOS calculator!

In [2]:
def run_section05c_checkpoints() -> None:
    '''Step-by-step checkpoint verifier for all 14 #TODO functions in Section 05c (Periodic Electronic Structure).'''
    passed = 0
    total_todos = 14

    try:
        a_mat = np.diag([4.60, 20.0, 20.0])
        b_mat, vol = compute_reciprocal_lattice_and_volume(a_mat)
        assert np.isclose(vol, 4.60 * 400.0) and np.allclose(a_mat @ b_mat.T, 2.0 * np.pi * np.eye(3))
        passed += 1
        print(f"[PASS] TODO 1 : compute_reciprocal_lattice_and_volume (b_1x = {b_mat[0,0]:.5f} Bohr^-1)")

        k_frac, k_cart, k_w = generate_monkhorst_pack_kmesh(b_mat, (8, 1, 1))
        assert k_cart.shape == (8, 3) and np.isclose(np.sum(k_w), 1.0) and np.allclose(np.sum(k_cart, axis=0), 0.0)
        passed += 1
        print("[PASS] TODO 2 : generate_monkhorst_pack_kmesh")

        k_gamma = np.array([0.0, 0.0, 0.0])
        k_x = np.array([np.pi / 4.60, 0.0, 0.0])
        kpath_c, kpath_d, ticks = interpolate_high_symmetry_kpath([-k_x, k_gamma, k_x], n_per_segment=25)
        assert kpath_c.shape[0] == 49 and len(ticks) == 3 and np.isclose(ticks[-1], 2.0 * np.pi / 4.60)
        passed += 1
        print("[PASS] TODO 3 : interpolate_high_symmetry_kpath")

        model = build_peierls_chain_real_space_tensors(a_lat=4.60, delta_dimer=0.15)
        S_gamma = bloch_lattice_sum_matrix(model["S_R"], model["lattice_translations"], k_gamma)
        S_kx = bloch_lattice_sum_matrix(model["S_R"], model["lattice_translations"], k_x)
        assert np.allclose(S_gamma, S_gamma.conj().T) and np.allclose(S_kx, S_kx.conj().T)
        passed += 1
        print("[PASS] TODO 4 : bloch_lattice_sum_matrix (Hermiticity verified across Brillouin zone)")

        H_gamma = bloch_lattice_sum_matrix(model["H_core_R"], model["lattice_translations"], k_gamma)
        eps_g, C_g, D_g = solve_kpoint_roothaan_hall(H_gamma, S_gamma, n_occ=1)
        assert np.isclose(2.0 * np.trace(D_g @ S_gamma).real, 2.0, atol=1e-12)
        passed += 1
        print("[PASS] TODO 5 : solve_kpoint_roothaan_hall")

        D_k_all = np.zeros((8, 2, 2), dtype=np.complex128)
        S_k_all = np.zeros((8, 2, 2), dtype=np.complex128)
        for ik in range(8):
            S_k_all[ik] = bloch_lattice_sum_matrix(model["S_R"], model["lattice_translations"], k_cart[ik])
            H_k = bloch_lattice_sum_matrix(model["H_core_R"], model["lattice_translations"], k_cart[ik])
            _, _, D_k_all[ik] = solve_kpoint_roothaan_hall(H_k, S_k_all[ik], n_occ=1)
        D_R_0 = reconstruct_real_space_density_from_kpts(D_k_all, k_cart, model["lattice_translations"], k_w)
        n_cell_elec = np.sum(2.0 * D_R_0 * model["S_R"])
        assert D_R_0.shape == (3, 2, 2) and np.isclose(n_cell_elec, 2.0, atol=1e-10)
        passed += 1
        print(f"[PASS] TODO 6 : reconstruct_real_space_density_from_kpts (Unit-cell electrons = {n_cell_elec:.8f})")

        assert model["H_core_R"].shape == (3, 2, 2) and np.allclose(model["H_core_R"][0], model["H_core_R"][2].T)
        passed += 1
        print("[PASS] TODO 7 : build_peierls_chain_real_space_tensors")

        F_g, Hc_g, JK_g = build_kpoint_fock_matrix(
            k_gamma, model["H_core_R"], model["V_ee_R"], D_R_0, model["lattice_translations"]
        )
        assert np.allclose(F_g, F_g.conj().T)
        passed += 1
        print("[PASS] TODO 8 : build_kpoint_fock_matrix")

        e_el_test, e_tot_test = compute_krhf_energy_per_unit_cell(D_k_all, S_k_all, S_k_all, k_w, 0.0)
        assert np.isclose(e_el_test, 2.0, atol=1e-10)
        passed += 1
        print("[PASS] TODO 9 : compute_krhf_energy_per_unit_cell")

        krhf_res = run_periodic_krhf_scf(model, n_kpts_1d=12, n_occ=1)
        assert krhf_res["converged"] and krhf_res["D_R"][1, 0, 1] > krhf_res["D_R"][2, 0, 1]
        passed += 1
        print(f"[PASS] TODO 10: run_periodic_krhf_scf (Converged E_cell = {krhf_res['e_tot_cell']:.8f} Eh in {krhf_res['n_iterations']} iters)")

        bands = compute_band_structure_along_kpath(kpath_c, model, krhf_res["D_R"])
        assert bands.shape == (49, 2) and np.all(bands[:, 1] > bands[:, 0])
        passed += 1
        print("[PASS] TODO 11: compute_band_structure_along_kpath")

        bg_info = analyze_band_gap_and_effective_mass(kpath_c[:, 0], bands, n_occ=1)
        assert bg_info["gap_ev"] > 0.5 and bg_info["is_direct_gap"] and bg_info["m_eff_cb"] > 0.0
        passed += 1
        print(f"[PASS] TODO 12: analyze_band_gap_and_effective_mass (Direct Band Gap E_g = {bg_info['gap_ev']:.3f} eV at k = {bg_info['k_vbm']:.4f})")

        e_grid = np.linspace(-25.0, 10.0, 200)
        dos = compute_electronic_density_of_states(krhf_res["eps_kpts"], krhf_res["k_weights"], e_grid, sigma_ev=0.3)
        de = e_grid[1] - e_grid[0]
        assert np.isclose(np.sum(dos) * de, 4.0, atol=0.05)
        passed += 1
        print(f"[PASS] TODO 13: compute_electronic_density_of_states (Integrated total states = {np.sum(dos)*de:.3f} / 4.0)")

        pbc_script = generate_pyscf_pbc_krhf_script()
        assert "KRHF" in pbc_script and "get_bands" in pbc_script
        passed += 1
        print("[PASS] TODO 14: generate_pyscf_pbc_krhf_script")
        print(f"\nCONGRATULATIONS! All {passed}/{total_todos} Section 05c (Periodic Electronic Structure) checkpoints passed!")

    except NotImplementedError as exc:
        print(f"\n[PROGRESS: {passed}/{total_todos} checkpoints passed] Next up -> {exc}")
    except AssertionError as exc:
        print(f"\n[ASSERTION FAILED after {passed}/{total_todos} checkpoints] -> {exc}")


# Uncomment and run as you implement each TODO:
# run_section05c_checkpoints()
print("Section 05c checkpoint suite loaded. Implement TODOs 1-14 and call `run_section05c_checkpoints()`!")